Theorems (or conjectures) for the theory of <a class="ProveItLink" href="theory.ipynb">proveit.physics.quantum.QEC2</a>
========

In [1]:
import proveit
# Prepare this notebook for defining the theorems of a theory:
%theorems_notebook # Keep this at the top following 'import proveit'.

from proveit            import b, c, e, f, m, n, p, s, A, B, G, X, ExprTuple, Function
from proveit.logic      import And, Equals, Exists, Forall, InSet, Implies
from proveit.logic.sets import (Card, Disjoint, EmptySet, KPowerSet, Set, SetOfAll, SubsetEq,
                                SymmetricDifference, Union, Difference, Intersect)
from proveit.graphs     import EdgeSequence, Graphs, IsGraph, IsPath, Paths, PathsOf, Subgraph
from proveit.numbers    import (zero, one, two, Add, frac, greater_eq,
                                LessEq, Mult, Natural, Neg)

from proveit.physics.quantum.QEC2 import (
        _ell, _end, _max_buf_weight, _nu, _start, ActionFunction, all_states_graph,
        all_states_graph_def, AllStatesGraph, BufiloGeneratingGraph, BufiloSequences,
        BufiloSets, CheckFunction, Errors, f_one_to_card_b, f_one_to_n, Faults,
        IrreducibleBufiloSets, m_prime, MalignantSets, ObservableSets, Realizations,
        State, StateAction, States, StateSyndrome, Weight)


In [2]:
%begin theorems

Defining theorems for theory 'proveit.physics.quantum.QEC2'
Subsequent end-of-cell assignments will define theorems
'%end theorems' will finalize the definitions


#### BUFILOs & BUFILO SETS

In [3]:
bufs_membership_unfolding = (
    Forall(b,
           And(InSet(b, Errors),
                    Equals(CheckFunction(b), EmptySet),
                    Equals(ActionFunction(_ell, b), one)),
    domain=BufiloSets)
)

In [4]:
bufs_membership_folding = (
    Forall(b,
           InSet(b, BufiloSets),
    conditions=[InSet(b, Errors), Equals(CheckFunction(b), EmptySet),
                Equals(ActionFunction(_ell, b), one)])
)

#### Irreducible BUFILOs (in terms of membership)

In [5]:
from proveit.logic import NotExists
from proveit.logic.sets import SubsetProper
from proveit.physics.quantum.QEC2 import b_prime
irreducible_bufs_membership_def = (
    Forall(b,
           Equals(InSet(b, IrreducibleBufiloSets),
                   And(InSet(b, BufiloSets),
                       NotExists(b_prime, SubsetProper(b_prime, b),
                                 domain=BufiloSets)
                   )
           )
    )
)

In [6]:
irreducible_bufs_membership_unfolding = (
    Forall(b,
           And(InSet(b, BufiloSets),
               NotExists(b_prime, SubsetProper(b_prime, b),
                         domain=BufiloSets)
           ),
    domain=IrreducibleBufiloSets)
)

In [7]:
irreducible_bufs_membership_folding = (
    Forall(b, InSet(b, IrreducibleBufiloSets),
    conditions=[InSet(b, BufiloSets),
                NotExists(b_prime, SubsetProper(b_prime, b),
                          domain=BufiloSets)])
)

#### Errors (sets of Faults)

In [8]:
errors_membership_unfolding = (
    Forall(e,
           Exists(n,
                         Exists((f_one_to_n),
                                Equals(e, Set(f_one_to_n)),
                         domain=Faults),
                  domain=Natural),
    domain=Errors)
)

In [9]:
errors_membership_folding = (
    Forall(e,
           InSet(e, Errors),
    conditions=[Exists(n, Exists((f_one_to_n),
                Equals(e, Set(f_one_to_n)),
                domain=Faults),
                domain=Natural)])
)

In [10]:
err_elem_is_fault = (
    Forall(e, Forall(f, InSet(f, Faults), domain=e), domain=Errors)
)

In [11]:
fault_set_is_error = Forall(n, Forall(f_one_to_n, InSet(Set(f_one_to_n), Errors), domain=Faults), domain=Natural)

#### Facts About Sets

In [12]:
difference_is_subset_of_symmetric_difference = (
    Forall((A, B),
           SubsetEq(Difference(A, B), SymmetricDifference(A, B))
    )
)

#### Facts About Weight

In [13]:
binary_disjoint_weight_additivity = (
    Forall((A, B),
           Equals(Weight(Union(A, B)),
                          Add(Weight(A), Weight(B))),
           conditions = [Disjoint(A, B)]
    )
)

In [14]:
weight_of_differences_inequality = (
    Forall((A, B),
           greater_eq(Weight(Difference(A, B)),
                      Weight(Difference(B, A))),
           conditions = [greater_eq(Weight(A), Weight(B))]
    )
)

#### Choice Function $\nu(s)$

In [15]:
nu_start_is_ell = Equals(Function(_nu, _start), _ell)

#### Malignant Set, $m$

We let $\textrm{MALS}$ denote the set of all possible _malignant sets_ (of faults) of a minimum-weight decoder.

Generally, a malignant set $m \in \textrm{MALS}$ is any set of faults that causes (or can cause) the QEC system to experience a logical failure. In the language and notation of Beverland, _et al._ (2025), a malignant set $m$ is one such that $H(m + c) = 0$ while $A(m + c) \ne 0$, where $H \in \mathbb{F}_{2}^{M \times N}$ is the “check matrix”, $A \in \mathbb{F}_{2}^{K \times N}$ is the “action matrix”, and $c = \mathcal{C}(\sigma)$ is the correction provided by the minimum-weight decoding algorithm $\mathcal{C}$.

Considered as a _set_ (instead of a Beverland vector), a malignant set $m$ is characterized by the fact that, for any such $m$, there exists both a correction $c$ and BUFILO $b$ such that $w(c) \le w(m)$ and $m \Delta c = b$, as captured more formally in the following theorem:

In [16]:
mal_set_property = (
    Forall(m,
       Exists(c,
       Exists(b, Equals(SymmetricDifference(m, c), b),
              domain=IrreducibleBufiloSets),
       conditions=[LessEq(Weight(c), Weight(m))]),
    domain=MalignantSets)
)

#### Theorem 1

##### For a minimum-weight decoder, every malignant fault set contains a non-minority subset of some BUFILO.

In [17]:
mal_set_contains_non_minority_subset_of_bufilo = Forall(m,
       Exists((m_prime, b),
       greater_eq(Weight(m_prime), Mult(frac(one, two), Weight(b))),
       conditions=[SubsetEq(m_prime, m), InSet(b, IrreducibleBufiloSets), SubsetEq(m_prime, b)]),
domain=MalignantSets)

#### Theorem 2
##### The sets of faults generated by $\mathcal{F}_{\ell,w_{\text{BUF}}}^{\text{seq}}$ exactly matches the BUFILOs up to weight $w_{\text{BUF}}$.

NOTE: In the formulation below, the `ExprTuple` construct is not formatting correctly, which makes the tuple $(f_{1}, f_{2},\ldots, f_{n})$ _appear_ as individual items $f_{1}, f_{2}, \ldots, f_{n}$.

In [18]:
from proveit.physics.quantum.QEC2 import Faults
from proveit.logic.sets import UnionAll

In [19]:
# bufilos_from_buf_seqs = Forall(n,
#        Equals(SetOfAll(ExprTuple(f_one_to_n), Set(f_one_to_n),
#                        condition=InSet(ExprTuple(f_one_to_n), BufiloSequences),
#                        domain = Faults),
#               SetOfAll(b, b, conditions=[LessEq(Weight(b), _max_buf_weight)], domain=BufiloSets)),
# domain = Natural)

In [20]:
# Theorem (1)
bufilos_from_buf_seqs = Equals(
    # LHS
    UnionAll(n, SetOfAll(ExprTuple(f_one_to_n), Set(f_one_to_n),
                       condition=InSet(ExprTuple(f_one_to_n), BufiloSequences),
                       domain = Faults),
          domain = Natural),
    # RHS
    SetOfAll(b, b, conditions=[LessEq(Weight(b), _max_buf_weight)], domain=BufiloSets)
)

ALTERNATIVE approach, WW thinks is more promising and comprehensive:

For each $\nu: \mathcal{S} \rightarrow \mathcal{D}$ ($\nu$ maps syndromes to an element of the syndrome; it's a "choice function" of a sort --- (include the “choice function” characteristic as an extra condition)), and for each BUFILO $b$, there exists $n$ and $f_{1}, f_{2}, \ldots, f_{n}$ such that $\{f_{1}, f_{2}, \ldots, f_{n}\} = b$ and:

$\forall_{i \in \{1,\ldots, n-1\}}$:

(1) $A_{l} \{f_{1}, \ldots, f_{i}\} = 0 \,\land \{f_{i+1}, \ell\} = 0$

or (2) $A_{l} \{f_{1}, \ldots, f_{i}\} \ne 0 \,\land \nu(H \{f_{1}, \ldots, f_{i}\}) \in H \{f_{i+1}\}$

Putting this together, we have:

$\forall_{\nu:[\mathcal{S}\rightarrow\mathcal{D}] \,|\, \forall_{X}\big(\nu(X) \in X\big)} \forall_{b \in \text{BUF}}
\Big[
\exists_{f_{1}, f_{2}, \ldots, f_{n} \,|\, n \in \mathbb{N}}
\Big[$

$\Big(\{f_{1}, f_{2}, \ldots, f_{n}\} = b \big)
\,\land\,$

$\forall_{i \in \{1,\ldots, n-1\}}\Big[\big[A_{l} \{f_{1}, \ldots, f_{i}\} = 0 \,\land [f_{i+1}, \ell]_{+} = 0\big] \lor \big[A_{l} \{f_{1}, \ldots, f_{i}\} \ne 0 \,\land \nu(H \{f_{1}, \ldots, f_{i}\}) \in H \{f_{i+1}\}\big]\Big]$

$\Big]\Big]$

In [21]:
from proveit import f, i, n, A, S, D, X, Y, Function, IndexedVar
from proveit.logic import And, Exists, NotEquals, Or
from proveit.logic.sets import IsFunction
from proveit.numbers import zero, one, Interval, subtract
from proveit.linear_algebra import AntiCommutator
from proveit.physics.quantum.QEC2 import ActionFunction, CheckFunction, Detectors, _ell, f_one_to_i, nu, Syndromes

In [22]:
Forall(nu,
       Forall(b,
              Exists((f_one_to_n),
                     And(
                         Equals(Set(f_one_to_n), b),
                         Forall(i,
                            Or(And(Equals(ActionFunction(_ell, Set(f_one_to_i)), zero),
                                   Equals(AntiCommutator(IndexedVar(f, Add(i, one)), _ell), zero)),
                               And(NotEquals(ActionFunction(_ell, Set(f_one_to_i)), zero),
                                   InSet(Function(nu, CheckFunction(Set(f_one_to_i))),
                                         CheckFunction(Set(IndexedVar(f, Add(i, one))))))),
                            domain=Interval(one, subtract(n, one))).with_wrapping()).with_wrap_after_operator(),
              conditions=[InSet(n, Natural)]).with_wrapping(),
       domain = BufiloSets).with_wrapping(),
conditions = [IsFunction(nu, Syndromes, Detectors), Forall(A, InSet(Function(nu, A), A))]).with_wrapping()

forall_{nu : [Syndromes -> Detectors] | forall_{A} (nu(A) in A)} [forall_{b in BUFS} [exists_{f_{1}, f_{2}, ..., f_{n} | n in Natural} (({f_{1}, f_{2}, ..., f_{n}} = b) and  \\ [forall_{i in {1 .. n - 1}} (((A_{l}({f_{1}, f_{2}, ..., f_{i}}) = 0) and ({f_{i + 1}, l'} = 0)) or ((A_{l}({f_{1}, f_{2}, ..., f_{i}}) != 0) and (nu(H({f_{1}, f_{2}, ..., f_{i}})) in H({f_{i + 1}}))))])]]

A possible supporting theorem:

Given an error $e$ such that $[e, \ell]_{+} = 0$ (_i.e._, $e$ anti-commutes with $\ell$), then $e$ “contains” a fault $f$ such that $[f, \ell]_{+} = 0$ (_i.e._, $f$ also anti-commutes with $\ell$).

We _can_ think of an error $e$ as a set of faults, but we've also defined an error $e$ as an $N \times 1$ binary vector: $e \in \mathbb{F}_{2}^{N}$

In [23]:
from proveit import e, f, X
from proveit.logic import Exists, Implies
from proveit.numbers import zero
from proveit.linear_algebra import AntiCommutator
from proveit.physics.quantum.QEC2 import _ell
Implies(Equals(AntiCommutator(e, _ell), zero),
        Exists(f, Equals(AntiCommutator(f, _ell), zero), domain = e))

({e, l'} = 0) => [exists_{f in e} ({f, l'} = 0)]

#### Detectors

In [24]:
emptyset_subset_of_detectors = SubsetEq(EmptySet, Detectors)

#### Syndromes & SyndromesMembership

In [25]:
syndromes_membership_unfolding = (
    Forall(s, SubsetEq(s, Detectors),
          domain=Syndromes)
)

In [26]:
syndromes_membership_folding = (
    Forall(s, InSet(s, Syndromes),
          conditions=[SubsetEq(s, Detectors)])
)

In [27]:
emptyset_is_syndrome = InSet(EmptySet, Syndromes)

In [28]:
empty_set_is_obs_set = InSet(EmptySet, ObservableSets)

In [29]:
start_is_obs_set = InSet(_start, ObservableSets)

In [30]:
from proveit.physics.quantum.QEC2 import ObservableSet
error_syndrome_in_obs_sets = Forall(e, InSet(CheckFunction(e), ObservableSets), domain=Errors)

In [31]:
fault_syndrome_in_obs_sets = Forall(f, InSet(CheckFunction(Set(f)), ObservableSets), domain=Faults)

In [32]:
obs_set_of_error_in_obs_sets = Forall(e, InSet(ObservableSet(e, _ell), ObservableSets), domain=Errors)

In [33]:
from proveit.physics.quantum.QEC2 import ObservableSet
obs_set_of_fault_in_obs_sets = Forall(f, InSet(ObservableSet(Set(f), _ell), ObservableSets), domain=Faults)

In [34]:
start_disjoint_from_error_syndrome = Forall(e, Disjoint(_start, CheckFunction(e)), domain=Errors)

In [35]:
start_disjoint_from_fault_syndrome = Forall(f, Disjoint(_start, CheckFunction(Set(f))), domain=Faults)

#### CheckFunction(), ActionFunction()

In [36]:
check_fxn_of_e_is_syndrome = Forall(e, InSet(CheckFunction(e), Syndromes), domain=Errors)

In [37]:
action_fxn_of_e_is_0_or_1 = Forall(e, InSet(ActionFunction(_ell, e), Set(zero, one)), domain=Errors)

#### Augmented Syndrome States and States Membership

Notice here that an explicit state such as $(D, i)$, perhaps created using the State class constructor $\text{State}(D, i)$, is not necessarily itself a legitimate State.

In [38]:
states_membership_unfolding = (
    Forall(s,
           And(InSet(StateSyndrome(s), Syndromes),
               InSet(StateAction(s), Set(zero, one))),
    domain=States)
)

In [39]:
states_membership_folding = (
    Forall(s,
           InSet(s, States),
    conditions=[And(InSet(StateSyndrome(s), Syndromes),
                InSet(StateAction(s), Set(zero, one)))])
)

In [40]:
states_membership_tuple_def = (
    Forall((D, i, s),
           Equals(InSet(State(D, i), States),
                 And(InSet(D, Syndromes), InSet(i, Set(zero, one))))
    )
)

In [41]:
states_membership_tuple_unfolding = (
    Forall((D, i),
           And(InSet(D, Syndromes), InSet(i, Set(zero, one))),
    conditions=[InSet(State(D, i), States)])
)

In [42]:
states_membership_tuple_folding = (
    Forall((D, i),
           InSet(State(D, i), States),
    conditions=[InSet(D, Syndromes), InSet(i, Set(zero, one))])
)

In [43]:
from proveit.physics.quantum.QEC2 import ErrorState
state_membership_unfolding = (
    Forall(s,
           Exists(e, Equals(s, ErrorState(_ell, e)), domain = Errors),
           domain=States)
)

#### ObservableSets and ObservableSetsMembership

In [44]:
from proveit.physics.quantum.QEC2 import ObservableSets
observable_sets_membership_unfolding = (
    Forall(s,
           SubsetEq(s, Union(Detectors, Set(_ell))),
    domain=ObservableSets)
)

In [45]:
observable_sets_membership_folding = (
    Forall(s,
           InSet(s, ObservableSets),
    conditions=[SubsetEq(s, Union(Detectors, Set(_ell)))])
)

####  EdgeFaults & EdgeFaultsMembership

In [46]:
from proveit import G
from proveit.numbers import Mod
from proveit.graphs import IsGraph
from proveit.physics.quantum.QEC2 import EdgeFaults, s_prime, StateAction, StateSyndrome
edge_faults_membership_unfolding = (
    Forall(G,
    Forall((s, s_prime),
               Forall(f, 
                      And(Equals(StateSyndrome(s_prime),
                                        SymmetricDifference(StateSyndrome(s), CheckFunction(Set(f)))),
                                 Equals(StateAction(s_prime),
                                        Mod(Add(StateAction(s), ActionFunction(_ell, Set(f))), two))).with_wrap_after_operator(),
               domain=EdgeFaults(s, s_prime, G)),
    domain=States),
    conditions=[IsGraph(G)])
)

In [47]:
edge_faults_membership_folding = (
    Forall(G,
    Forall((s, s_prime),
           Forall(f, 
                  InSet(f, EdgeFaults(s, s_prime, G)),
           conditions=[And(Equals(StateSyndrome(s_prime),
                                  SymmetricDifference(StateSyndrome(s), CheckFunction(Set(f)))),
                           Equals(StateAction(s_prime),
                                  Mod(Add(StateAction(s), ActionFunction(_ell, Set(f))), two)))]),
    domain=States),
    conditions=[IsGraph(G)])
)

#### Realizations & RealizationsMembership

In [48]:
from proveit import ExprRange
from proveit.graphs import Edges
from proveit.physics.quantum.QEC2 import e_one_to_n, e_i, f_i, Realizations
realizations_membership_unfolding = (
    Forall(G,
    Forall(n,
    Forall((e_one_to_n),
             Forall(f_one_to_n,
                    And(ExprRange(i, InSet(f_i, EdgeFaults(e_i, G)),
                                         one, n)),
             conditions=[InSet(ExprTuple(f_one_to_n),
                                 Realizations(ExprTuple(e_one_to_n), G))],
             domain=Faults).with_wrapping(),
    domain=Edges(G)).with_wrapping(),
    domain = Natural).with_wrapping(),
    conditions=[IsGraph(G)]).with_wrapping()
)

In [49]:
realizations_membership_folding = (
    Forall(G,
    Forall(n,
    Forall((e_one_to_n),
             Forall(f_one_to_n,
                    InSet(ExprTuple(f_one_to_n),
                                 Realizations(ExprTuple(e_one_to_n), G)),
             conditions=[And(ExprRange(i, InSet(f_i, EdgeFaults(e_i, G)),
                                         one, n))],
             domain=Faults).with_wrapping(),
    domain=Edges(G)).with_wrapping(),
    domain = Natural).with_wrapping(),
    conditions=[IsGraph(G)]).with_wrapping()
)

#### Anticommuting Error $e$ Contains Anticommuting Fault

In [50]:
anticommuting_error_contains_anticommuting_fault = (
    Forall(e, Exists(f, Equals(ActionFunction(_ell, Set(f)), one), domain=e),
    conditions=[Equals(ActionFunction(_ell, e), one)], domain=Errors)
)

### All-States Graph `AllStatesGraph` & Related

In [51]:
# The following is needed for invoking some vertex and edge-related machinery
# involving the AllStatesGraph
all_states_graph_verts, all_states_graph_edges = (
        all_states_graph_def.rhs.operands[0],
        all_states_graph_def.rhs.operands[1])
all_states_graph_edges_two_elem_subsets_of_nodes = SubsetEq(all_states_graph_edges, KPowerSet(all_states_graph_verts, two))

In [52]:
exists_buf_gen_graph = Forall(b, 
       Exists(p,
              Exists(f_one_to_card_b,
                     And(InSet(ExprTuple(f_one_to_card_b), Realizations(p, AllStatesGraph)),
                         Equals(b, Set(f_one_to_card_b))),
                     domain=Faults).with_wrapping(),
       conditions=[IsPath(p, AllStatesGraph, State(EmptySet, zero), State(EmptySet, one))]).with_wrapping(),
conditions=[LessEq(Weight(b), _max_buf_weight)], domain=IrreducibleBufiloSets).with_wrapping()

In [53]:
exists_buf_gen_graph_alt = Forall(b, 
       Exists(p,
              Exists(f_one_to_card_b,
                     And(InSet(ExprTuple(f_one_to_card_b), Realizations(EdgeSequence(p), AllStatesGraph)),
                         Equals(b, Set(f_one_to_card_b))),
                     domain=Faults).with_wrapping(),
       conditions=[InSet(p, PathsOf(AllStatesGraph, begin=State(EmptySet, zero), end=State(EmptySet, one)))]).with_wrapping(),
conditions=[LessEq(Weight(b), _max_buf_weight)], domain=IrreducibleBufiloSets).with_wrapping()

#### BUFILO-Generating Graph $G_{\ell \nu}^{BUF}$

In [54]:
buf_gen_graph_complete = (
    Forall(b, 
           Exists(p,
                  Exists(f_one_to_card_b,
                         And(InSet(ExprTuple(f_one_to_card_b), Realizations(EdgeSequence(p), BufiloGeneratingGraph)),
                             Equals(b, Set(f_one_to_card_b))),
                         domain=Faults).with_wrapping(),
           conditions=[InSet(p, PathsOf(BufiloGeneratingGraph, begin=_start, end=_end))]).with_wrapping(),
    conditions=[LessEq(Weight(b), _max_buf_weight)], domain=IrreducibleBufiloSets).with_wrapping()

)

The version below, labeled slightly differently as `buf_gen_graph_completeness` (instead of using “complete”), uses `RealizationSets` instead of `Realizations`.

In [55]:
from proveit.physics.quantum.QEC2 import RealizationSets
buf_gen_graph_completeness = (
    Forall(b, 
           Exists(p,
                  InSet(b, RealizationSets(EdgeSequence(p), BufiloGeneratingGraph)),
           conditions=[InSet(p, PathsOf(BufiloGeneratingGraph, begin=_start, end=_end))]).with_wrapping(),
    conditions=[], domain=IrreducibleBufiloSets).with_wrapping()
)

Assigned after/during discussion Fri 8/28/2026:

(2) Think about the induction statement in the proof of the above. Might need a finite version of induction (see induction-related theorems in `numbers/numbersets/naturals`). We'd likely prove something along the lines of $\forall_{j \le n}$, which would then prove the statement for the existent $n \in \mathbb{N}$.

(3) Continue formulating the theorems in the paper, and edit paper to reflect the Prove-It versions of these theorems, along with some contextual text to explain.

Proof of above?

To form a BUFILO, every detector must (eventually) be turned off.

That gives us flexibility in how to order the faults!

Proof by induction:

$b = \{f_{1}, f_{2}, \ldots, f_{n}\}$

Begin: there exists $f$ such that $A_{\ell}(\{f\}) =  1$

Given that we've chosen $f_{1}, f_{2}, \ldots, f_{j}$ faults for fault sequence …

Case (1) empty syndrome? (for $j < n$) Either we've finished a BUFILO (not possible, since we can't have a BUFILO proper-subset), or we have a homologically-trivial cycle (for example, an even number of BUFILOs).

Case (2) non-empty syndrome. For all active detectors, among the remaining $(n-j)$ faults there must be at least one fault that eliminates that detector. Thus there must exist an edge e in the graph where $f_{j+1} \in EdgeFaults(e)$.

In [56]:
%end theorems

These theorems may now be imported from the theory package: proveit.physics.quantum.QEC2
